# 04 — Player-Level Roster Update

A transparent ingestion and aggregation workflow for official final-squad player records. The repository does not bundle proprietary live minutes, market values, or form feeds, so this notebook validates an optional user-supplied player CSV and falls back to the released team-level final-26 features when the player table is absent.

**Repository:** SambaSportAI World Cup 2026 Forecasting Pipeline  
**Execution:** Run from the repository root or directly from the `notebooks/` folder.  
**Reproducibility:** All inputs are read from version-controlled CSV/JSON artifacts in this repository.

In [1]:
from pathlib import Path
import json
import math
import ast
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    # Common fallback when launched from notebooks/ via Jupyter.
    fallback = Path.cwd().resolve().parent
    if (fallback / "pyproject.toml").exists():
        return fallback
    raise FileNotFoundError("Could not locate repository root. Launch Jupyter inside the repository.")

ROOT = find_repo_root()
print(f"Repository root: {ROOT}")

Repository root: /mnt/data/sambasportai-world-cup-2026


## 1. Data boundary and expected input

The released repository contains team-level final-squad features in:

`data/baseline/v8_final26_team_ratings.csv`

A true player-level update requires an additional file:

`data/raw/player_level_final26.csv`

The file should contain one row per called-up player. The notebook deliberately does **not** invent missing player records or scrape licensing-restricted sources.

In [2]:
team_ratings_path = ROOT / "data" / "baseline" / "v8_final26_team_ratings.csv"
player_path = ROOT / "data" / "raw" / "player_level_final26.csv"
template_path = ROOT / "data" / "raw" / "player_level_final26_template.csv"

team_ratings = pd.read_csv(team_ratings_path)
print(f"Team-level ratings: {team_ratings.shape}")
display(team_ratings.head())

Team-level ratings: (48, 19)


,team,base_hybrid_rating,base_xg_for,base_xg_against,official_final_squad_status,official_squad_size,source_basis,availability_delta,attack_delta,defense_delta,goalkeeping_delta,roster_rating_delta,roster_note,roster_strength_index,attack_strength_index,defense_strength_index,roster_coverage_flag,squad_size_numeric,trained_roster_rating
0,Mexico,1960.082673,2.192828,0.900986,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.015,0.005,0.015,0.0,0.01275,Guillermo Ochoa included for a sixth World Cup...,1965.117673,2.208828,-0.879986,1,26,1974.367910
1,South Africa,1680.904741,1.063222,1.986729,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,1680.904741,1.063222,-1.986729,1,26,1688.626494
2,South Korea,1899.123239,1.480654,1.489421,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.015,0.010,0.000,0.0,0.01125,Final squad includes dual-heritage addition Je...,1903.298239,1.502904,-1.484171,1,26,1900.163788
3,Czech Republic,1799.015595,1.296645,1.656213,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,1799.015595,1.296645,-1.656213,1,26,1801.606675
4,Canada,1881.715329,1.902187,0.945367,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,-0.095,-0.050,-0.055,0.0,-0.07825,"Injury-hit hosts: Alphonso Davies sidelined, M...",1851.435329,1.777937,-1.036367,1,26,1854.763896


## 2. Player-level schema and template

In [3]:
required_columns = [
    "team", "player_name", "position", "date_of_birth", "club", "league",
    "squad_number", "is_final_squad", "is_available",
    "club_minutes_2025_26", "club_appearances_2025_26",
    "goals_2025_26", "assists_2025_26",
    "market_value_eur", "form_rating",
    "source_name", "source_url", "retrieved_at"
]

schema = pd.DataFrame({
    "column": required_columns,
    "description": [
        "Canonical national-team name",
        "Player's full name",
        "GK / DF / MF / FW",
        "ISO date YYYY-MM-DD",
        "Current club",
        "Current domestic league",
        "Tournament squad number",
        "1 if named in the final squad",
        "1 if currently available, 0 if ruled out",
        "Club minutes in the 2025/26 season",
        "Club appearances in the 2025/26 season",
        "Club goals in the 2025/26 season",
        "Club assists in the 2025/26 season",
        "Market value in euros from a licensed/public source",
        "Optional normalized recent-form rating",
        "Source publisher/provider",
        "Source URL or dataset identifier",
        "Retrieval timestamp",
    ]
})
display(schema)

if not template_path.exists():
    pd.DataFrame(columns=required_columns).to_csv(template_path, index=False)
    print(f"Created empty template: {template_path.relative_to(ROOT)}")
else:
    print(f"Template already exists: {template_path.relative_to(ROOT)}")

,column,description
0,team,Canonical national-team name
1,player_name,Player's full name
2,position,GK / DF / MF / FW
3,date_of_birth,ISO date YYYY-MM-DD
4,club,Current club
5,league,Current domestic league
6,squad_number,Tournament squad number
7,is_final_squad,1 if named in the final squad
8,is_available,"1 if currently available, 0 if ruled out"
9,club_minutes_2025_26,Club minutes in the 2025/26 season


Template already exists: data/raw/player_level_final26_template.csv


## 3. Validation functions

In [4]:
VALID_POSITIONS = {"GK", "DF", "MF", "FW"}

def validate_player_table(players):
    errors = []
    warnings_list = []

    missing = sorted(set(required_columns) - set(players.columns))
    if missing:
        errors.append(f"Missing required columns: {missing}")
        return errors, warnings_list

    duplicates = players.duplicated(["team", "player_name"], keep=False)
    if duplicates.any():
        errors.append(f"Duplicate team/player rows: {duplicates.sum()}")

    invalid_positions = sorted(set(players["position"].dropna().astype(str)) - VALID_POSITIONS)
    if invalid_positions:
        errors.append(f"Invalid positions: {invalid_positions}")

    final_players = players[players["is_final_squad"].fillna(0).astype(int) == 1]
    squad_sizes = final_players.groupby("team").size()
    bad_sizes = squad_sizes[(squad_sizes < 23) | (squad_sizes > 26)]
    if len(bad_sizes):
        warnings_list.append("Teams outside the expected 23–26 final-squad range: " + bad_sizes.to_dict().__repr__())

    nonnegative_columns = [
        "club_minutes_2025_26", "club_appearances_2025_26", "goals_2025_26",
        "assists_2025_26", "market_value_eur"
    ]
    for col in nonnegative_columns:
        numeric = pd.to_numeric(players[col], errors="coerce")
        if (numeric.dropna() < 0).any():
            errors.append(f"Negative values found in {col}")

    missing_source = players["source_name"].isna() | players["source_url"].isna()
    if missing_source.any():
        warnings_list.append(f"Rows without complete provenance: {missing_source.sum()}")

    return errors, warnings_list

## 4. Load optional player data or use the team-level fallback

In [5]:
if player_path.exists():
    players = pd.read_csv(player_path)
    errors, validation_warnings = validate_player_table(players)
    print(f"Loaded player table: {players.shape}")
    print("Errors:", errors or "None")
    print("Warnings:", validation_warnings or "None")
    if errors:
        raise ValueError("Player-level input failed validation. Correct the errors before aggregation.")
    player_data_available = True
else:
    players = pd.DataFrame(columns=required_columns)
    player_data_available = False
    display(Markdown(
        "**Player-level input is not bundled.** The notebook will continue with the released "
        "team-level final-26 ratings. Populate `data/raw/player_level_final26.csv` using the template "
        "and rerun the notebook to activate player aggregation."
    ))

**Player-level input is not bundled.** The notebook will continue with the released team-level final-26 ratings. Populate `data/raw/player_level_final26.csv` using the template and rerun the notebook to activate player aggregation.

## 5. Aggregate player-level features

In [6]:
POSITION_WEIGHTS = {"GK": 0.80, "DF": 0.90, "MF": 1.00, "FW": 1.05}

def minmax(series):
    series = pd.to_numeric(series, errors="coerce")
    if series.notna().sum() == 0:
        return pd.Series(np.nan, index=series.index)
    lo, hi = series.min(), series.max()
    if hi == lo:
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)

def aggregate_player_features(players):
    frame = players.copy()
    frame = frame[frame["is_final_squad"].fillna(0).astype(int) == 1].copy()
    frame["available"] = frame["is_available"].fillna(1).astype(float)
    frame["minutes_norm"] = minmax(frame["club_minutes_2025_26"]).fillna(0.0)
    frame["value_norm"] = minmax(frame["market_value_eur"]).fillna(0.0)
    frame["form_norm"] = minmax(frame["form_rating"]).fillna(0.0)
    frame["production"] = (
        pd.to_numeric(frame["goals_2025_26"], errors="coerce").fillna(0)
        + 0.7 * pd.to_numeric(frame["assists_2025_26"], errors="coerce").fillna(0)
    )
    frame["production_norm"] = minmax(frame["production"]).fillna(0.0)
    frame["position_weight"] = frame["position"].map(POSITION_WEIGHTS).fillna(1.0)

    frame["player_strength"] = frame["available"] * frame["position_weight"] * (
        0.35 * frame["minutes_norm"]
        + 0.30 * frame["value_norm"]
        + 0.20 * frame["form_norm"]
        + 0.15 * frame["production_norm"]
    )

    team = (
        frame.groupby("team")
        .agg(
            squad_size=("player_name", "size"),
            available_players=("available", "sum"),
            total_minutes=("club_minutes_2025_26", "sum"),
            total_market_value_eur=("market_value_eur", "sum"),
            mean_form_rating=("form_rating", "mean"),
            mean_player_strength=("player_strength", "mean"),
            top11_strength=("player_strength", lambda s: s.nlargest(min(11, len(s))).mean()),
            top5_strength=("player_strength", lambda s: s.nlargest(min(5, len(s))).mean()),
        )
        .reset_index()
    )
    team["availability_rate"] = team["available_players"] / team["squad_size"]
    return team, frame

if player_data_available:
    player_team_features, enriched_players = aggregate_player_features(players)
    display(player_team_features.sort_values("top11_strength", ascending=False).head(15))
else:
    player_team_features = pd.DataFrame()
    enriched_players = pd.DataFrame()
    display(team_ratings[[
        "team", "trained_roster_rating", "roster_strength_index", "attack_strength_index",
        "defense_strength_index", "availability_delta", "roster_note"
    ]].sort_values("trained_roster_rating", ascending=False).head(15))

,team,trained_roster_rating,roster_strength_index,attack_strength_index,defense_strength_index,availability_delta,roster_note
30,Spain,2223.704686,2227.881631,2.893898,-0.740747,0.000,Final squad published; no specific additional ...
36,Argentina,2177.427013,2180.705879,2.790282,-0.726589,0.020,Lionel Messi included for another World Cup ca...
32,France,2131.063583,2140.082166,2.227741,-0.984056,0.000,Final squad published; no specific additional ...
44,England,2088.563380,2088.180281,2.458363,-0.853590,0.000,Final squad published; no specific additional ...
8,Brazil,2061.631409,2061.472938,2.585338,-0.909699,0.025,"Neymar returned to the final squad, with fitne..."
40,Portugal,2036.910504,2038.316188,2.229231,-0.994737,0.020,Cristiano Ronaldo included in final squad.
20,Netherlands,2026.724968,2021.494010,2.372317,-0.794339,0.000,Final squad published; no specific additional ...
43,Colombia,2019.455711,2026.292934,1.832631,-1.252470,0.000,Final squad published; no specific additional ...
16,Germany,2013.920024,2008.734410,2.334751,-0.912059,0.000,Final squad published; no specific additional ...
19,Ecuador,1997.961034,2007.724677,1.627082,-1.467981,0.000,Final squad published; no specific additional ...


## 6. Merge with released team ratings

In [7]:
def build_updated_team_ratings(team_ratings, player_team_features, blend_weight=0.20):
    base = team_ratings.copy()
    if player_team_features.empty:
        base["player_update_available"] = 0
        base["player_level_strength"] = np.nan
        base["updated_team_rating"] = base["trained_roster_rating"]
        return base

    update = player_team_features.copy()
    # Standardize player strength and map it to rating-point scale.
    z = (update["top11_strength"] - update["top11_strength"].mean()) / update["top11_strength"].std(ddof=0)
    update["player_level_strength"] = 100 * z

    merged = base.merge(update[["team", "player_level_strength", "availability_rate", "total_market_value_eur"]], on="team", how="left")
    merged["player_update_available"] = merged["player_level_strength"].notna().astype(int)
    merged["player_level_strength"] = merged["player_level_strength"].fillna(0.0)
    merged["updated_team_rating"] = (
        (1 - blend_weight) * merged["trained_roster_rating"]
        + blend_weight * (merged["trained_roster_rating"] + merged["player_level_strength"])
    )
    return merged

updated_team_ratings = build_updated_team_ratings(team_ratings, player_team_features)
display(updated_team_ratings.sort_values("updated_team_rating", ascending=False).head(20))

,team,base_hybrid_rating,base_xg_for,base_xg_against,official_final_squad_status,official_squad_size,source_basis,availability_delta,attack_delta,defense_delta,goalkeeping_delta,roster_rating_delta,roster_note,roster_strength_index,attack_strength_index,defense_strength_index,roster_coverage_flag,squad_size_numeric,trained_roster_rating,player_update_available,player_level_strength,updated_team_rating
30,Spain,2227.881631,2.893898,0.740747,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2227.881631,2.893898,-0.740747,1,26,2223.704686,0,NaN,2223.704686
36,Argentina,2173.540879,2.746032,0.733589,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.020,0.025,0.000,0.0,0.01850,Lionel Messi included for another World Cup ca...,2180.705879,2.790282,-0.726589,1,26,2177.427013,0,NaN,2177.427013
32,France,2140.082166,2.227741,0.984056,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2140.082166,2.227741,-0.984056,1,26,2131.063583,0,NaN,2131.063583
44,England,2088.180281,2.458363,0.853590,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2088.180281,2.458363,-0.853590,1,26,2088.563380,0,NaN,2088.563380
8,Brazil,2051.662938,2.512838,0.907949,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.025,0.045,-0.010,0.0,0.02525,"Neymar returned to the final squad, with fitne...",2061.472938,2.585338,-0.909699,1,26,2061.631409,0,NaN,2061.631409
40,Portugal,2031.151188,2.184981,1.001737,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.020,0.025,0.000,0.0,0.01850,Cristiano Ronaldo included in final squad.,2038.316188,2.229231,-0.994737,1,26,2036.910504,0,NaN,2036.910504
20,Netherlands,2021.494010,2.372317,0.794339,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2021.494010,2.372317,-0.794339,1,26,2026.724968,0,NaN,2026.724968
43,Colombia,2026.292934,1.832631,1.252470,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2026.292934,1.832631,-1.252470,1,26,2019.455711,0,NaN,2019.455711
16,Germany,2008.734410,2.334751,0.912059,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2008.734410,2.334751,-0.912059,1,26,2013.920024,0,NaN,2013.920024
19,Ecuador,2007.724677,1.627082,1.467981,final squad published,up to 26,Reuters final squad list + FIFA/Wikipedia squa...,0.000,0.000,0.000,0.0,0.00000,Final squad published; no specific additional ...,2007.724677,1.627082,-1.467981,1,26,1997.961034,0,NaN,1997.961034


## 7. Export optional updated features

In [8]:
output_path = ROOT / "data" / "processed" / "player_level_team_features.csv"
ratings_output_path = ROOT / "data" / "processed" / "team_ratings_with_player_update.csv"

if player_data_available:
    player_team_features.to_csv(output_path, index=False)
    updated_team_ratings.to_csv(ratings_output_path, index=False)
    print(f"Saved: {output_path.relative_to(ROOT)}")
    print(f"Saved: {ratings_output_path.relative_to(ROOT)}")
else:
    print("No player-level export was written because data/raw/player_level_final26.csv is absent.")

No player-level export was written because data/raw/player_level_final26.csv is absent.


## Responsible use

- Use only data sources whose terms permit research and redistribution.
- Record a source name, URL/dataset identifier, and retrieval time for every player row.
- Do not treat market value as a direct measure of human worth or as a complete measure of football ability.
- Availability, club minutes, and recent form are time-sensitive; freeze all inputs at a declared prediction cutoff.
- Re-run held-out validation before replacing released model weights with player-level updates.